In [1]:
from pathlib import Path
import requests

from openmm import *
from openmm.app import *
from openmm import unit
from openmm.unit import *

import mdtraj as md

import pdbfixer
from pdbfixer import PDBFixer

from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem

from pdbecif.mmcif_io import CifFileReader
from pdbecif.mmcif_io import CifFileWriter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nglview as nv

import MDAnalysis as mda
from MDAnalysis.analysis import rms, diffusionmap, align
from MDAnalysis.analysis.distances import dist
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import HydrogenBondAnalysis as HBA

In [ ]:
HERE = Path(_dh[-1])
DATA = HERE / "data"

In [ ]:
pdb_id = "6LUQ"
url = (f"https://www.ebi.ac.uk/pdbe/entry-files/download/{pdb_id.lower()}.cif")
response = requests.get(url)
response.raise_for_status()

destination = DATA / f"{pdb_id}.cif"
with open(destination, "wb") as file:
    file.write(response.content)

print(f"Saved structure to {destination}")

In [ ]:
# Setting the force field
forcefield = ForceField('amber19-all.xml', 'amber19/tip3pfb.xml')

In [ ]:
# Read mmCIF file into PDBxFile object
pdbx = PDBxFile((DATA / '6LUQ.cif').as_posix())

In [ ]:
def prepare_protein(
    pdb_file, ignore_missing_residues=True, ignore_terminal_missing_residues=True, ph=7.4
):
    """
    Use pdbfixer to prepare the protein from a PDB file. Hetero atoms such as ligands are
    removed and non-standard residues replaced. Missing atoms to existing residues are added.
    Missing residues are ignored by default, but can be included.

    Parameters
    ----------
    pdb_file: pathlib.Path or str
        PDB file containing the system to simulate.
    ignore_missing_residues: bool, optional
        If missing residues should be ignored or built.
    ignore_terminal_missing_residues: bool, optional
        If missing residues at the beginning and the end of a chain should be ignored or built.
    ph: float, optional
        pH value used to determine protonation state of residues

    Returns
    -------
    fixer: pdbfixer.pdbfixer.PDBFixer
        Prepared protein system.
    """
    fixer = pdbfixer.PDBFixer(str(pdb_file))
    fixer.removeHeterogens()  # remove co-crystallized ligands, water molecules and other non-polymers
    fixer.findMissingResidues()  # identify missing residues, needed for identification of missing atoms

    # if missing terminal residues shall be ignored, remove them from the dictionary
    if ignore_terminal_missing_residues:
        chains = list(fixer.topology.chains())
        keys = fixer.missingResidues.keys()
        for key in list(keys):
            chain = chains[key[0]]
            if key[1] == 0 or key[1] == len(list(chain.residues())):
                del fixer.missingResidues[key]

    # if all missing residues shall be ignored ignored, clear the dictionary
    if ignore_missing_residues:
        fixer.missingResidues = {}

    fixer.findNonstandardResidues()  # find non-standard residue
    fixer.replaceNonstandardResidues()  # replace non-standard residues with standard one
    fixer.findMissingAtoms()  # find missing heavy atoms
    fixer.addMissingAtoms()  # add missing atoms and residues
    fixer.addMissingHydrogens(ph)  # add missing hydrogens
    return fixer

In [ ]:
# prepare protein and build only missing non-terminal residues
prepared_protein = prepare_protein((DATA / '6LUQ.cif').as_posix(), ignore_missing_residues=False)
print('Successfully prepared protein')

In [ ]:
def get_ligandatoms_fromstructuremodel(mmciffile, pdbid, lig_entity_id):
    data = CifFileReader().read(mmciffile, only=['_atom_site'])
    atom_site = data[pdbid]['_atom_site']

    lig_atoms = {key: [] for key in atom_site.keys()}
    number_of_atoms = len(atom_site['id'])
    for i in range(number_of_atoms):
        if atom_site['label_entity_id'][i] == str(lig_entity_id):
            for key in atom_site:
                lig_atoms[key].append(atom_site[key][i])
            
    print(f"Extracted {len(lig_atoms['id'])} ligand atoms (entity {lig_entity_id})")
    return lig_atoms


def fix_numeric_types(atom_site):
    float_fields = ['Cartn_x', 'Cartn_y', 'Cartn_z', 'occupancy', 'B_iso_or_equiv']
    for field in float_fields:
        if field in atom_site:
            atom_site[field] = [float(x) for x in atom_site[field]]
    return atom_site


def write_mmcif_ligandatoms(mmciffile, pdbid, lig_entity_id, outp_file):
    lig_atoms = get_ligandatoms_fromstructuremodel(mmciffile, pdbid, lig_entity_id)
    lig_atoms = fix_numeric_types(lig_atoms)
    # print(lig_atoms)
    datablock = {pdbid: {'_atom_site': lig_atoms}}
    cfw = CifFileWriter(outp_file)
    cfw.write(datablock)
    return outp_file

In [ ]:
write_mmcif_ligandatoms((DATA / '6LUQ.cif').as_posix(), 
                        '6LUQ', 3, 
                        (DATA / 'ligandatoms.cif').as_posix())

In [ ]:
ligand_id = "03P"
url = (f"https://files.rcsb.org/ligands/download/{ligand_id}_ideal.sdf")

response = requests.get(url)
response.raise_for_status()
destination = DATA / f"{ligand_id}_ideal.sdf"
with open(destination, "wb") as f:
    f.write(response.content)

print(f"Saved ligand to {destination}")

In [ ]:
def prepare_ligand():
    # Ideal ligand
    supplier = Chem.SDMolSupplier(str(DATA / "03P_ideal.sdf"), removeHs=False)
    template_lig = supplier[0]
    if template_lig is None:
        raise ValueError("Failed to read ligand SDF")
    template_lig = Chem.RemoveHs(template_lig)
    
    # Ligand from experimental structure model  
    ligandatoms = write_mmcif_ligandatoms(str(DATA / "3POZ.cif"), '3POZ', 3, str(DATA / "lig_from_experiment.cif"))
    # As we have mmCIF format, but rdkit cannot handle them, we first convert the file to (legacy) PDB format
    fixer = pdbfixer.PDBFixer(str(DATA / "lig_from_experiment.cif"))
    PDBFile.writeFile(fixer.topology, fixer.positions, open(str(DATA / "lig_from_experiment.pdb"), 'w'))

    # Load into rdkit so we can work with it
    experimental_lig = Chem.MolFromPDBFile(str(DATA / "lig_from_experiment.pdb"), removeHs=True)
    # experimental_lig = Chem.RemoveHs(experimental_lig)
    if experimental_lig is None:
        raise ValueError("Failed to read ligand from coordinate file")
    
    # Prepare ligand for MD
    if len(template_lig.GetAtoms()) != len(experimental_lig.GetAtoms()):
        print("Template atoms:", [a.GetSymbol() for a in template_lig.GetAtoms()])
        print("Target atoms:", [a.GetSymbol() for a in experimental_lig.GetAtoms()])
        raise ValueError("Atom counts mismatch! Cannot assign bond orders.")    
    prepared_ligand = AllChem.AssignBondOrdersFromTemplate(template_lig, experimental_lig)   
    prepared_ligand.AddConformer(experimental_lig.GetConformer(0), assignId=True)
    prepared_ligand = Chem.AddHs(prepared_ligand, addCoords=True)
    AllChem.ComputeGasteigerCharges(prepared_ligand) #Look into, do we need these charges computed?
    
    Chem.MolToPDBFile(prepared_ligand, str(DATA / "ligand_prepared.pdb"))

    return prepared_ligand

In [ ]:
# Prepare ligand
prepared_ligand = prepare_ligand()
print('Successfully prepared ligand')
prepared_ligand

In [ ]:
# Load pre-generated ligand parameters into the forcefield (so we redefine the forcefield here, 
# if you changed the forcefield at the beginning of this tutorial, make sure to do that here as well)
forcefield = ForceField("amber14-all.xml", "amber14/tip3pfb.xml", str(DATA / "6LUQ.ffxml"))

In [ ]:
# Add the protein
modeller = Modeller(prepared_protein.topology, prepared_protein.positions)
modeller.addHydrogens(forcefield, pH=7.4) #We do this before adding the ligand, as the ligand already has hydrogen atoms
# Add ligand
ligand_pdb = PDBFile(str(DATA /'ligand_prepared.pdb'))
modeller.add(ligand_pdb.topology, ligand_pdb.positions)
# Add solvent
modeller.addSolvent(forcefield, padding=1.0 * openmm.unit.nanometer, ionicStrength=0.15 * openmm.unit.molar)

In [ ]:
system = forcefield.createSystem(modeller.topology, nonbondedMethod=PME)
integrator = LangevinIntegrator(
    300 * openmm.unit.kelvin, 1.0 / openmm.unit.picoseconds, 2.0 * openmm.unit.femtoseconds
)
simulation = Simulation(modeller.topology, system, integrator)
simulation.context.setPositions(modeller.positions)

In [ ]:
%%time 

print('Minimizing...')
# simulation.minimizeEnergy() #Takes 30-35 minutes in noteable AY2627
simulation.minimizeEnergy(maxIterations=1000) #Takes about 15 minutes in noteable AY2627

with open(DATA / "topology.pdb", "w") as pdb_file:
    PDBFile.writeFile(simulation.topology, simulation.context.getState(getPositions=True, enforcePeriodicBox=True).getPositions(),
                      file=pdb_file,keepIds=True)

print('Finalized the minimization')

In [ ]:
# To make sure we start with a fresh logfile each time running the talktorial
logfile = DATA / "simulation_log.txt"
if logfile.exists():
    logfile.unlink()

In [ ]:
# To make sure we start with a fresh logfile each time running the talktorial
trajectoryfile = DATA / "trajectory.xtc"
if trajectoryfile.exists():
    trajectoryfile.unlink()

In [ ]:
# Note that for usage in Noteable we shortened the simulation time even further, 
# if you still think this is too long, you can reduce the values below yourself
steps = 10000               #25000 == 25k steps × 2 fs = 50 ps
write_interval = 250        #500 == write every 1 ps
log_interval = 250          #500 == log every 5 ps
simulation.reporters = []   # to start with a clear reporter each simulation

steps = 25000           # 25k steps × 2 fs = 50 ps
write_interval = 500    # write every 1 ps
log_interval = 500     # log every 5 ps
simulation.reporters = []   # to start with a clear reporter each simulation
simulation.reporters.append(md.reporters.XTCReporter(file=str(DATA/"trajectory.xtc"), reportInterval=write_interval))
simulation.reporters.append(StateDataReporter(str(DATA/"simulation_log.txt"), log_interval,step=True,potentialEnergy=True,temperature=True,
                                              progress=True,remainingTime=True,speed=True,totalSteps=steps,separator="\t"))

In [ ]:
%%time 
#Takes XXX minutes in noteable AY2627

# Simulate
print('Simulating...')
simulation.context.setVelocitiesToTemperature(300 * openmm.unit.kelvin)
simulation.step(steps)  # perform the simulation
print('Done simulating!')

In [ ]:
# Check the trajectory exists and is not empty
(DATA / "trajectory.xtc").stat().st_size > 0

In [ ]:
# Check the logfile exists and is not empty
(DATA / "simulation_log.txt").stat().st_size > 0

In [ ]:
# Check how the logfile looks like
with open(DATA / "simulation_log.txt") as f:
    for _ in range(10):
        print(f.readline())

In [ ]:
# Plotting the temperature over time
df = pd.read_csv(logfile, sep="\t")

plt.plot(df["Step"], df["Temperature (K)"])
plt.xlabel("Step")
plt.ylabel("Temperature (K)")
plt.show()

In [ ]:
# Plotting the potential energy over time
plt.plot(df["Step"], df["Potential Energy (kJ/mole)"])
plt.xlabel("Step")
plt.ylabel("Potential Energy (kJ/mol)")
plt.show()

In [ ]:
md_universe = mda.Universe(str(DATA / "topology.pdb"), str(DATA / "trajectory.xtc"))
view = nv.show_mdanalysis(md_universe)
view

In [ ]:
# In case the above view does not allow for playing the movie, the play-button below 
# can be used to click through the trajectory and mimic the movie functionality
display(view._iplayer)

In [ ]:
# Set trajectory pointer to first frame as reference
md_universe.trajectory[0]

# in_memory=True is needed to actually modify the loaded trajectory
# if the trajectory is too big for memory, one can write out the aligned trajectory and reload it into a new universe
alignment = align.AlignTraj(
    mobile=md_universe, reference=md_universe, select="protein", in_memory=True
)
alignment.run()

In [ ]:
view = nv.show_mdanalysis(md_universe)
view

In [ ]:
# In case the above view does not allow for playing the movie, the play-button below 
# can be used to click through the trajectory and mimic the movie functionality
display(view._iplayer)

In [ ]:
ligand_name = "03P"

In [ ]:
def rmsd_for_atomgroups(universe, selection1, selection2=None):
    """Calulate the RMSD for selected atom groups.

    Parameters
    ----------
    universe: MDAnalysis.core.universe.Universe
        MDAnalysis universe.
    selection1: str
        Selection string for main atom group, also used during alignment.
    selection2: list of str, optional
        Selection strings for additional atom groups.

    Returns
    -------
    rmsd_df: pandas.core.frame.DataFrame
        DataFrame containing RMSD of the selected atom groups over time.
    """

    universe.trajectory[0]
    ref = universe
    rmsd_analysis = rms.RMSD(universe, ref, select=selection1, groupselections=selection2)
    rmsd_analysis.run()
    columns = [selection1, *selection2] if selection2 else [selection1]
    rmsd_df = pd.DataFrame(np.round(rmsd_analysis.rmsd[:, 2:], 2), columns=columns)
    rmsd_df.index.name = "frame"
    return rmsd_df

In [ ]:
rmsd = rmsd_for_atomgroups(md_universe, "backbone", ["protein", f"resname {ligand_name}"])
rmsd.head()

In [ ]:
rmsd.plot(title="RMSD of protein and ligand")
plt.ylabel("RMSD (Å)");

In [ ]:
def RMSD_dist_frames(universe, selection):
    """Calculate the RMSD between all frames in a matrix.

    Parameters
    ----------
    universe: MDAnalysis.core.universe.Universe
        MDAnalysis universe.
    selection: str
        Selection string for the atomgroup to be investigated, also used during alignment.

    Returns
    -------
    array: np.ndarray
        Numpy array of RMSD values.
    """
    pairwise_rmsd = diffusionmap.DistanceMatrix(universe, select=selection)
    pairwise_rmsd.run()
    return pairwise_rmsd.dist_matrix

In [ ]:
dist_matrix_protein = RMSD_dist_frames(md_universe, "protein")
dist_matrix_ligand = RMSD_dist_frames(md_universe, f"resname {ligand_name}")

In [ ]:
# get maximal distance between all frames
max_dist = max(np.amax(dist_matrix_ligand), np.amax(dist_matrix_protein))

In [ ]:
fig, ax = plt.subplots(1, 2)
fig.suptitle("RMSD between the frames")

# protein image
img1 = ax[0].imshow(dist_matrix_protein, cmap="viridis", vmin=0, vmax=max_dist)
ax[0].title.set_text("protein")
ax[0].set_xlabel("frames")
ax[0].set_ylabel("frames")

# ligand image
img2 = ax[1].imshow(dist_matrix_ligand, cmap="viridis", vmin=0, vmax=max_dist)
ax[1].title.set_text("Ligand")
ax[1].set_xlabel("frames")

fig.colorbar(img1, ax=ax, orientation="horizontal", fraction=0.1, label="RMSD (Å)")

In [ ]:
# select pocket residues
pocket_resids = set(
    md_universe.select_atoms(f"(around 5 resname {ligand_name}) and protein").resids
)
view2 = nv.show_mdanalysis(md_universe)
view2.add_representation(
    "licorice", selection=f"protein and ({' '.join([str(x) for x in pocket_resids])})"
)
view2.center(selection="ligand")
view2

In [ ]:
# In case the above view does not allow for playing the movie, the play-button below 
# can be used to click through the trajectory and mimic the movie functionality
display(view2._iplayer)


In [ ]:
protein_residue_id = "793"
protein_atom_name = "N"
ligand_id = "1"
ligand_atom_name = "N"

In [ ]:
atomgroup_protein = md_universe.select_atoms(
    f"resid {protein_residue_id} and name {protein_atom_name}"
)
atomgroup_ligand = md_universe.select_atoms(f"resname {ligand_name} and name {ligand_atom_name}")

In [ ]:
distance = dist(atomgroup_protein, atomgroup_ligand)
print(
    f"The distance between residues {protein_residue_id} and {ligand_id} is {distance[2][0]:.2f} Å"
)

In [ ]:
distances = []
for _ in md_universe.trajectory:
    distance = dist(atomgroup_protein, atomgroup_ligand)
    distances.append(distance[2][0])

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(distances)
plt.gca().set_ylim(bottom=0)
plt.ylabel("distance (Å)")
plt.xlabel("frame")
plt.title(f"Atomic distance between atoms of protein residue {protein_residue_id}-{protein_atom_name} and ligand {ligand_name} {ligand_id}-{ligand_atom_name}")
plt.show()

In [ ]:
hydrogen_name = "H"

In [ ]:
def run_hydrogenbond_analysis(
    universe,
    donor_resid,
    donor_atom,
    hydrogen_atom,
    acceptor_resid,
    acceptor_atom,
    h_cutoff=1.2,
    a_cutoff=3.0,
    angle_cutoff=120,
):
    """
    Perform an hydrogen bond analysis between selcted atom groups.
    Return the data as readable, formatted data frame.

    Parameters
    ----------
    universe: MDAnalysis.core.universe.Universe
        MDAnalysis universe.
    donor_resid: str
        Residue Id of residue containing donor atom in MDAnalysis universe.
    donor_atom: str
        Name of the donor atom in the MDAnalysis universe.
    hydrogen_atom: str
        Name of involved hydrogen atom in the MDAnalysis universe.
    acceptor_resid: str
        Residue Id of residue containing acceptor atom in the MDAnalysis universe.
    acceptor_atom: str
        Name of the acceptor atom in the MDAnalysis universe.
    h_cutoff: float, optional
        Distance cutoff used for finding donor-hydrogen pairs.
    a_cutoff: float, optional
        Donor-acceptor distance cutoff for hydrogen bonds.
    angle_cutoff: int, optional
        D-H-A angle cutoff for hydrogen bonds.

    Returns
    -------
    bond : pandas.core.frame.DataFrame
        DataFrame containing hydrogen bond information over time.
    """

    # set up and run the hydrogen bond analysis (HBA)
    hbonds = HBA(
        universe=universe,
        donors_sel="resid " + donor_resid + " and name " + donor_atom,
        hydrogens_sel="resid " + donor_resid + " and name " + hydrogen_atom,
        acceptors_sel="resid " + acceptor_resid + " and name " + acceptor_atom,
        d_h_cutoff=h_cutoff,
        d_a_cutoff=a_cutoff,
        d_h_a_angle_cutoff=angle_cutoff,
    )
    hbonds.run()
    # extract bond data
    bond = pd.DataFrame(np.round(hbonds.hbonds, 2))
    # name columns (according to MDAnalysis function description)
    bond.columns = [
        "frame",
        "donor_index",
        "hydrogen_index",
        "acceptor_index",
        "distance",
        "angle",
    ]
    # set frame as the index
    bond["frame"] = bond["frame"].astype(int)
    bond.set_index("frame", inplace=True, drop=True)
    return bond

In [ ]:
bond = run_hydrogenbond_analysis(
    md_universe, protein_residue_id, protein_atom_name, hydrogen_name, ligand_id, ligand_atom_name
)
bond.head()

In [ ]:
bond = run_hydrogenbond_analysis(
    md_universe,
    protein_residue_id,
    protein_atom_name,
    hydrogen_name,
    ligand_id,
    ligand_atom_name,
    a_cutoff=5.0,
)
bond.head()

In [ ]:
def plot_hydrogen_bond_over_time(bond):
    """
    Plot the distance and degree of a chosen hydrogen bond over time.
    Peaks in angle and dips in distance represent moments of strong binding.

    Parameters
    ----------
    bond: pandas.core.frame.DataFrame
        Hydrogen bond analysis data frame with named columns.
    """

    donor_index = bond.iloc[0].donor_index
    acceptor_index = bond.iloc[0].acceptor_index

    # add values for missing frames for gapless plotting
    bond = bond.reindex(np.arange(0, 99))

    # plot the data
    fig = plt.figure(figsize=(10, 6))
    distance = bond.distance.plot(
        title=f"Hydrogen bond between protein residue {protein_residue_id} and {ligand_name} ({ligand_atom_name})",
        style="o-",
        x="frame",
    )
    distance.set_ylabel("distance (Å)")
    distance.set_xlabel("frame")
    distance.set_ylim(bottom=0)
    angle = bond.angle.plot(secondary_y=True, style="o-", x="frame")
    angle.set_ylabel("angle (°)")
    angle.set_xlabel("frame")
    angle.set_ylim(bottom=0)
    fig.legend(loc="lower right", bbox_to_anchor=(1, 0), bbox_transform=distance.transAxes)
    plt.show()

In [ ]:
plot_hydrogen_bond_over_time(bond)

In [ ]:
view_strong = nv.show_mdanalysis(md_universe)
view_strong.add_representation("ball+stick", selection=f"protein and {protein_residue_id}")
view_strong.center(selection="ligand")
view_strong.frame = 71
view_strong

In [ ]:
# In case the above view does not allow for playing the movie, the play-button below 
# can be used to click through the trajectory and mimic the movie functionality
display(view_strong._iplayer)

In [ ]:
view_bad = nv.show_mdanalysis(md_universe)
view_bad.add_representation("ball+stick", selection=f"protein and {protein_residue_id}")
view_bad.center(selection="ligand")
view_bad.frame = 1
view_bad

In [ ]:
# In case the above view does not allow for playing the movie, the play-button below 
# can be used to click through the trajectory and mimic the movie functionality
display(view_bad._iplayer)